**Hadamard transform**

- use a normalized hadamard transform to normalize lengths across matrix rows
- this makes it easier to quantize - because we remove the large outlier values in rows, that would otherwise destroy the precision of quantization
	- recall quantization uses a scale factor = max_value in a matrix / highest representable number in quantized format
	- e.g. max representable number in FP8_E4M3 is 448 
- given this we DO NOT want a big scale factor relative to our average matrix value. That would destroy precision (e.g. 0.22/5000 rounded to nearest FP8 representation destroys precision)

Deepseekv3 uses this transformation before quantizing a weight layer, and uses it during forward pass

$$XW \rightarrow (X \hat H^T) (\hat H W)$$

- where we quantize $\hat H W$ and calculate $ (X \hat H^T) $ on the forward pass

Where $\hat H = \dfrac{1}{\sqrt{d_{embed}}}H$
- this is the normalization factor
- the fast Hadamard transform is $O(nlogn)$, so it is a cheap way to get rid of outliers in a matrix

In [ ]:
def rotate_activation(x: torch.Tensor): 

	assert x.dtype == torch.bfloat16

	from fast_hadamard_transform import hadamard_transform
	
	hidden_dim = x.size(-1)
	# normalize the activation
	return hadamard_transform(x, scale=hidden_dim ** -0.5)